# 02 — Probe exercise: write the L1 core yourself

Three functions, ~30 lines total. The asserts fail until your implementations are
right; when they all pass, your code IS the method registered in
`PREREG-9B.md` Amendment 3. Your version cross-checks the pipeline I build for
`04-l1-probes` — if our AUROCs disagree on real data, one of us has a bug and we
find it together.

Rules of the exercise: numpy for `diff_in_means` and `grouped_auroc` (by hand —
no sklearn.metrics); sklearn's `LogisticRegression` IS allowed inside
`fit_probe_scores` (the craft there is the *pipeline*: standardize, grouped CV,
out-of-fold scores — not the optimizer).


In [ ]:
import numpy as np
rng = np.random.default_rng(7)

# Synthetic world shaped like our cache: resid[trial, round, d], a planted
# "drift" direction whose PRE-EVENT magnitude differs between leakers and
# holders (that's the signal the probe must find), plus a nuisance direction
# that varies with ROUND (that's the confound round-conditional eval defeats).
N_TRIALS, N_ROUNDS, D = 48, 8, 64
true_dir = rng.normal(size=D); true_dir /= np.linalg.norm(true_dir)
round_dir = rng.normal(size=D); round_dir /= np.linalg.norm(round_dir)

will_leak = np.arange(N_TRIALS) < N_TRIALS // 2          # first half leak
X = rng.normal(size=(N_TRIALS, N_ROUNDS, D))
for i in range(N_TRIALS):
    X[i] += np.arange(N_ROUNDS)[:, None] * 0.8 * round_dir      # round confound
    if will_leak[i]:
        X[i] += 2.0 * true_dir                                   # trait signal
print("X", X.shape, "| leakers:", int(will_leak.sum()))


## Task 1 — `diff_in_means`

Return the UNIT direction `mean(class1) − mean(class0)` from row vectors and
binary labels. (numpy only)


In [ ]:
def diff_in_means(H, y):
    """H: [n, d] activations; y: [n] bool. -> unit vector [d]."""
    raise NotImplementedError  # YOUR CODE (3-4 lines)


# -- checks --
# Pool rounds per trial: the round confound is the SAME for both classes,
# so it cancels in the class-mean difference — averaging rounds buys SNR free.
Hflat = X.mean(axis=1)
d_hat = diff_in_means(Hflat, will_leak)
assert d_hat.shape == (D,) and abs(np.linalg.norm(d_hat) - 1) < 1e-6
cos = float(d_hat @ true_dir)
print(f"cosine(d_hat, planted direction) = {cos:.2f}")
assert cos > 0.85, "should recover the planted direction at this SNR"
print("TASK 1 PASSED")


## Task 2 — `grouped_auroc`

AUROC from scores and labels, BY HAND from the rank definition: the fraction of
(positive, negative) pairs where the positive scores higher (ties count 0.5).
No sklearn.metrics. (The 'grouped' part of honesty lives in Task 3's CV; here
you build the metric itself so you know what the number is.)


In [ ]:
def auroc(scores, y):
    """scores: [n] float; y: [n] bool. -> float in [0, 1]."""
    raise NotImplementedError  # YOUR CODE (4-6 lines; O(n^2) loop is fine)


# -- checks --
s = np.array([0.9, 0.8, 0.7, 0.3, 0.2, 0.1])
yy = np.array([1, 1, 0, 1, 0, 0], bool)
val = auroc(s, yy)
assert abs(val - 8.5/9) < 1e-9, f"expected {8.5/9:.4f}, got {val}"
assert auroc(np.array([1., 1., 1., 1.]), np.array([1, 0, 1, 0], bool)) == 0.5
print("TASK 2 PASSED")


## Task 3 — `fit_probe_scores`

Out-of-fold probe scores with GROUPED cross-validation: standardize (fit the
scaler on train folds only!), logistic regression, and every trial's score must
come from a fold that never saw that trial. Return scores aligned to rows.
sklearn allowed: `StandardScaler`, `LogisticRegression`, `GroupKFold`.


In [ ]:
def fit_probe_scores(H, y, groups, n_splits=5):
    """H: [n, d]; y: [n] bool; groups: [n] trial ids.
    -> out-of-fold P(leak) scores [n]."""
    raise NotImplementedError  # YOUR CODE (~10 lines)


# -- checks: round-conditional eval on the synthetic world --
rows_H, rows_y, rows_g = [], [], []
for r in range(N_ROUNDS):
    rows_H.append(X[:, r, :]); rows_y.append(will_leak)
    rows_g.append(np.arange(N_TRIALS))
H_all = np.concatenate(rows_H); y_all = np.concatenate(rows_y)
g_all = np.concatenate(rows_g)

scores = fit_probe_scores(H_all, y_all, g_all)
a = auroc(scores, y_all)
a_shuf = auroc(fit_probe_scores(H_all, rng.permutation(y_all), g_all),
               rng.permutation(y_all))
print(f"probe AUROC = {a:.2f}   shuffled = {a_shuf:.2f}")
assert a > 0.75, "signal is planted; a correct pipeline finds it"
assert 0.35 < a_shuf < 0.65, "shuffled labels must sit near chance"
print("TASK 3 PASSED")


## Task 4 (the punchline) — see the confound with your own tools

The round direction is a CONFOUND: a probe comparing round-1 states to round-8
states can score well by reading the clock. Show it: AUROC of a probe trained
to separate (round ≤ 2) vs (round ≥ 7) states using only never-leakers — there
is NO trait signal there, yet the number will be high. Then explain in one
sentence in the cell below why Amendment 3's round-conditional negatives kill
this. (Write the 5 lines yourself with your Task-3 function.)


In [ ]:
# YOUR CODE: build H/y/groups for never-leakers, label = round>=7 vs <=2,
# run fit_probe_scores + auroc. Expect ~0.9+ despite zero trait signal.


**Your one-sentence explanation of why this is why round-conditional negatives exist:**

> (write here)

## Done?

When all tasks pass, run the last cell to check your functions against the real
smoke-cache trial's shapes (1 trial for now; the full cache lands tonight).


In [ ]:
import glob, json as js
fs = glob.glob("../microscope/cache/qwen35-9b-v1/*.npz")
if fs:
    z = np.load(fs[0]); meta = js.load(open(fs[0].replace(".npz", ".json")))
    print("real trial:", meta["persona"], meta["item_id"], meta["outcome"])
    print("resid:", z["resid"].shape, "-> layer 20 vector:", z["resid"][0, 20].shape)
    d20 = z["resid"][:, 20, :].astype(np.float32)
    print("your diff_in_means runs on real d=4096:",
          diff_in_means(np.vstack([d20, d20 + 1]),
                        np.array([False, True])).shape)
else:
    print("cache not on disk yet — rerun after tonight's Q0")
